In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

#### Load data

In [ ]:
with open("./data/results/retrieved_qu/ft_1803.jsonl", "r") as f:
    answers_ft = json.load(f)

with open(
    "./data/results/retrieved_qu/base_llama3.1:8b.jsonl", "r"
) as f:
    answers_base = json.load(f)
with open("./data/results/retrieved_qu/base_llama3.3.jsonl", "r") as f:
    base_33 = json.load(f)

path_base33_judge_mistral = (
    "./data/results/retrieved_qu/base_llama3.3_judge-mistral.jsonl"
)
with open(path_base33_judge_mistral, "r") as f:
    base_33_judge_mistral = json.load(f)

path_base33_judge_mistral_few_shots = "./data/results/retrieved_qu/base_llama3.3_judge-mistral_few_shots.jsonl"
with open(path_base33_judge_mistral_few_shots, "r") as f:
    base_33_judge_mistral_few_shots = json.load(f)

In [ ]:
base_33[1]

In [ ]:
np.sum(np.isnan([qu["rating"] for qu in base_33]))

In [ ]:
len(base_33)

#### Process results to clean and filter results

In [ ]:
df_ft = pd.DataFrame(answers_ft)[
    ["title", "question", "id", "RAG_answer", "answer", "rating"]
]
df_base = pd.DataFrame(answers_base)[["id", "RAG_answer", "rating"]]
df_base_33 = pd.DataFrame(base_33)[["id", "RAG_answer", "rating"]]

df_ft.set_index("id", inplace=True)
df_base.set_index("id", inplace=True)
df_base_33.set_index("id", inplace=True)

merged = df_ft.merge(df_base, how="inner", on="id", suffixes=("_ft", "_base"))
df_base_33.rename(
    columns={"RAG_answer": "RAG_answer_33", "rating": "rating33"}, inplace=True
)
merged = merged.merge(df_base_33, how="inner", on="id")

merged = merged[
    (merged["rating_ft"] <= 5)
    & (merged["rating_base"] <= 5)
    & (merged["rating33"] <= 5)
].dropna()
print(f"Len of test results after filtering: {len(merged)}")

print(f"Average rating for base model: {merged['rating_base'].mean()}")
print(f"Average rating for fine-tuned model: {merged['rating_ft'].mean()}")
print(f"Average rating for base model 3.3: {merged['rating33'].mean()}")

In [ ]:
def merge_list_results(results_list, names_list):
    df_list = []
    for i in range(len(results_list)):
        df = pd.DataFrame(results_list[i])[["id", "RAG_answer", "rating"]]
        df.rename(
            columns={
                "RAG_answer": f"RAG_answer_{names_list[i]}",
                "rating": f"rating_{names_list[i]}",
            },
            inplace=True,
        )
        df.set_index("id", inplace=True)
        df_list.append(df)
    merged = df_list[0]
    for i in range(1, len(df_list)):
        merged = merged.merge(df_list[i], how="inner", on="id")
    # filtering bad results
    for name in names_list:
        merged = merged[merged[f"rating_{name}"] <= 5]
    # print mean ratings
    for name in names_list:
        print(f"Average rating for {name}: {merged[f'rating_{name}'].mean()}")
    return merged

#### Bar plot

In [ ]:
df_melted = merged.melt(
    value_vars=["rating_base", "rating_ft", "rating33"],
    var_name="Model",
    value_name="Rating",
)

# Filter valid ratings (1-5)
df_melted = df_melted[df_melted["Rating"].between(1, 5)]

# Count occurrences of each rating for each model
df_counts = df_melted.groupby(["Rating", "Model"]).size().reset_index(name="Count")

# Create the bar plot
plt.figure(figsize=(8, 6))
sns.barplot(x="Rating", y="Count", hue="Model", data=df_counts, palette="Set2")

# Customization
plt.xlabel("Rating (1-5)")
plt.ylabel("Count")
plt.title("Distribution of Ratings for Each Model")
plt.legend(title="Model")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show the plot
plt.show()

In [ ]:
def print_bar_plot_merged_df(merged):
    colums_name_ratings = [col for col in merged.columns if "rating" in col]
    df_melted = merged.melt(
        value_vars=colums_name_ratings, var_name="Model", value_name="Rating"
    )

    # Filter valid ratings (1-5)
    df_melted = df_melted[df_melted["Rating"].between(1, 5)]

    # Count occurrences of each rating for each model
    df_counts = df_melted.groupby(["Rating", "Model"]).size().reset_index(name="Count")

    # Create the bar plot
    plt.figure(figsize=(8, 6))
    sns.barplot(x="Rating", y="Count", hue="Model", data=df_counts, palette="Set2")

    # Customization
    plt.xlabel("Rating (1-5)")
    plt.ylabel("Count")
    plt.title("Distribution of Ratings for Each Model")
    plt.legend(title="Model")
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    # Show the plot
    plt.show()

# Use here

In [ ]:
merged = merge_list_results(
    [
        answers_base,
        answers_ft,
        base_33,
        base_33_judge_mistral,
        base_33_judge_mistral_few_shots,
    ],
    [
        "base_llama3.1:8B",
        "FineTuned_llama3.1:8B",
        "llama3.3:70B",
        "llama3.3:70B_judge_mistral(24B)",
        "llama3.3:70B_judge_mistral(24B)_few_shots",
    ],
)
print_bar_plot_merged_df(merged)